Celda 1: Markdown - Título y Configuración Inicial
Parcial II Corte - Matemáticas Computacionales
Detección de anomalías en tráfico de red mediante modelos de espacio de estados y análisis espectral
Integrantes: [Nombres de los 5 integrantes]Docente: Udualdo Herrera PhD (c)

Este notebook contiene la implementación completa del sistema híbrido (SSM + Espectral + Kalman) para la detección de anomalías en el dataset UNSW-NB15.

In [1]:
# Instalar dependencias necesarias (ejecutar solo si faltan)
!pip install kagglehub[pandas-datasets] imbalanced-learn scikit-learn matplotlib imbalanced-ensemble

In [2]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np

# Set the path to the file you'd like to load
# Según la estructura del dataset, los archivos son CSV separados
file_path_train = "UNSW_NB15_training-set.csv"
file_path_test = "UNSW_NB15_testing-set.csv"

# Load the latest version
df_train = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "mrwellsdavid/unsw-nb15",
  file_path_train,
)

df_test = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "mrwellsdavid/unsw-nb15",
  file_path_test,
)

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)
df_train.head()

/tmp/ipykernel_14002/3739986652.py:12: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df_train = kagglehub.load_dataset(


Using Colab cache for faster access to the 'unsw-nb15' dataset.


/tmp/ipykernel_14002/3739986652.py:18: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df_test = kagglehub.load_dataset(


Using Colab cache for faster access to the 'unsw-nb15' dataset.
Train shape: (82332, 45)
Test shape: (175341, 45)


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.000011,udp,-,INT,2,0,496,0,90909.0902,...,1,2,0,0,0,1,2,0,Normal,0
1,2,0.000008,udp,-,INT,2,0,1762,0,125000.0003,...,1,2,0,0,0,1,2,0,Normal,0
2,3,0.000005,udp,-,INT,2,0,1068,0,200000.0051,...,1,3,0,0,0,1,3,0,Normal,0
3,4,0.000006,udp,-,INT,2,0,900,0,166666.6608,...,1,3,0,0,0,2,3,0,Normal,0
4,5,0.000010,udp,-,INT,2,0,2126,0,100000.0025,...,1,3,0,0,0,2,3,0,Normal,0


In [6]:
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from imblearn.pipeline import Pipeline as ImbPipeline # Changed from imbalanced_ensemble.pipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
import torch
from torch.utils.data import DataLoader, TensorDataset

# Variables categóricas y objetivo
cat_cols = ['proto', 'service', 'state']
target_col = 'label'
id_cols = ['id']

# Eliminar columnas que no usaremos
drop_cols = [c for c in df_train.columns if c not in cat_cols + [target_col] and df_train[c].dtype == 'object']
df_train = df_train.drop(columns=drop_cols + id_cols, errors='ignore')
df_test = df_test.drop(columns=drop_cols + id_cols, errors='ignore')

# Codificación de variables categóricas
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df_train[col] = le.fit_transform(df_train[col])
    df_test[col] = df_test[col].apply(lambda x: x if x in le.classes_ else -1)
    df_test[col] = le.transform(df_test[col].replace({-1: le.classes_[0]})) # Manejo de categorías desconocidas
    encoders[col] = le

# Separación X, y
X_train = df_train.drop(columns=[target_col]).values.astype(np.float32)
y_train = df_train[target_col].values.astype(np.int64)
X_test = df_test.drop(columns=[target_col]).values.astype(np.float32)
y_test = df_test[target_col].values.astype(np.int64)

# Normalización Min-Max (Ajustada solo en Train!)
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Ventaneo deslizante
def create_windows(X, y, window_size=10):
    X_w, y_w = [], []
    for i in range(len(X) - window_size):
        X_w.append(X[i:i+window_size])
        # Etiqueta de la ventana: la del último paso (o la mayoría, usamos el último por simplicidad y requisito SSM)
        y_w.append(y[i+window_size-1])
    return np.array(X_w), np.array(y_w)

window_size = 10
X_train_w, y_train_w = create_windows(X_train, y_train, window_size)
X_test_w, y_test_w = create_windows(X_test, y_test, window_size)

# Balanceo de clases (Aplanamos, aplicamos SMOTE+RUS, y rearmamos)
n_samples, n_timesteps, n_features = X_train_w.shape
X_train_flat = X_train_w.reshape(n_samples, n_timesteps * n_features)

# Pipeline de balanceo
resampler = ImbPipeline([
    ('oversample', SMOTE(sampling_strategy='minority', random_state=42)), # Changed from 0.5 to 'minority'
    ('undersample', RandomUnderSampler(sampling_strategy=1.0, random_state=42))
])

X_res_flat, y_res = resampler.fit_resample(X_train_flat, y_train_w)
X_res = X_res_flat.reshape(-1, n_timesteps, n_features)

print(f"Datos de entrenamiento balanceados: {X_res.shape}, Distribución de clases: {np.bincount(y_res)}")

# Creación de DataLoaders
train_dataset = TensorDataset(torch.tensor(X_res), torch.tensor(y_res))
test_dataset = TensorDataset(torch.tensor(X_test_w), torch.tensor(y_test_w))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

input_dim = n_features

Datos de entrenamiento balanceados: (90664, 10, 42), Distribución de clases: [45332 45332]


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SelectiveSSM(nn.Module):
    def __init__(self, d_model, d_state=16):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state

        # Parámetros SSM
        self.log_A = nn.Parameter(torch.log(torch.ones(d_state) * 0.5)) # Diagonal
        self.B_proj = nn.Linear(d_model, d_state)
        self.C_proj = nn.Linear(d_model, d_state)
        self.D_param = nn.Parameter(torch.ones(d_model))
        self.dt_proj = nn.Linear(d_model, 1)

    def forward(self, u):
        """
        u: (B, L, D)
        """
        B, L, D = u.shape
        N = self.d_state

        # A estable (Euler discretization)
        A = -torch.exp(self.log_A) # A < 0
        dt = F.softplus(self.dt_proj(u)) # (B, L, 1)
        dA = 1 + A * dt # (B, L, N) -> Broadcasteado

        B_t = self.B_proj(u) # (B, L, N)
        C_t = self.C_proj(u) # (B, L, N)

        # Recurrencia
        x = torch.zeros(B, N, device=u.device)
        ys = []

        for t in range(L):
            x = dA[:, t, :] * x + B_t[:, t, :] * dt[:, t, 0].unsqueeze(1)
            y = torch.einsum('bn,bn->b', x, C_t[:, t, :]) # Producto punto
            ys.append(y)

        y = torch.stack(ys, dim=1) # (B, L)
        return y.unsqueeze(-1) + self.D_param * u


class TemporalBlock(nn.Module):
    def __init__(self, d_model, d_state=16, kernel_size=3):
        super().__init__()
        self.conv1d = nn.Conv1d(d_model, d_model, kernel_size, padding=kernel_size-1, groups=d_model)
        self.ssm = SelectiveSSM(d_model, d_state)
        self.norm = nn.LayerNorm(d_model)
        self.silu = nn.SiLU()
        self.proj_in = nn.Linear(d_model, d_model * 2) # Gating

    def forward(self, x):
        residual = x
        x = self.conv1d(x.transpose(1, 2)).transpose(1, 2)[:, :x.size(1), :]
        x = self.silu(x)
        x_ssm = self.ssm(x)

        # Gating
        x_gate = self.proj_in(x_ssm)
        x1, x2 = x_gate.chunk(2, dim=-1)
        x = x1 * torch.sigmoid(x2)

        return self.norm(x + residual)


class SpectralBlock(nn.Module):
    def __init__(self, d_model, top_k=8):
        super().__init__()
        self.top_k = top_k
        # Filtro complejo aprendible por canal (inicializado en 1 + 0j)
        self.complex_filter = nn.Parameter(torch.ones(d_model, dtype=torch.cfloat))

    def forward(self, x):
        B, L, D = x.shape
        # rfft a lo largo del eje temporal
        X_f = torch.fft.rfft(x, dim=1) # (B, L/2+1, D) Complejo

        # Top-K componentes
        magnitudes = torch.abs(X_f)
        _, topk_idx = torch.topk(magnitudes, self.top_k, dim=1)
        mask = torch.zeros_like(magnitudes, dtype=torch.bool)
        mask.scatter_(1, topk_idx, True)

        # Filtrado
        X_f_filtered = X_f * mask * self.complex_filter.unsqueeze(0).unsqueeze(0)

        # Vuelta al dominio temporal
        x_out = torch.fft.irfft(X_f_filtered, n=L, dim=1)
        return x_out


class Fusion(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        # Inicialización: empezar como SSM puro
        self.alpha = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))

    def forward(self, x_temp, x_freq):
        return self.alpha * x_temp + self.beta * x_freq


class AnomalyDetector(nn.Module):
    def __init__(self, input_dim, d_model=64, d_state=16, top_k=8):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.temporal_block = TemporalBlock(d_model, d_state)
        self.spectral_block = SpectralBlock(d_model, top_k)
        self.fusion = Fusion(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.SiLU(),
            nn.Linear(d_model // 2, 2) # 2 clases (Normal, Ataque)
        )

    def forward(self, x):
        x = self.input_proj(x)
        x_temp = self.temporal_block(x)
        x_freq = self.spectral_block(x)
        z = self.fusion(x_temp, x_freq)
        logits = self.classifier(z)
        return logits

    def predict_proba(self, x):
        self.eval()
        with torch.no_grad():
            logits = self.forward(x)
            prob = torch.softmax(logits, dim=-1)
            return prob[..., 1] # Probabilidad de anomalía

    @classmethod
    def load(cls, path, input_dim, d_model=64, d_state=16, top_k=8):
        model = cls(input_dim, d_model, d_state, top_k)
        model.load_state_dict(torch.load(path))
        return model

In [9]:
class KalmanSmoother:
    def __init__(self, sigma_w2=1e-4, sigma_v2=1e-2):
        self.sigma_w2 = sigma_w2
        self.sigma_v2 = sigma_v2
        self.s_hat = 0.0 # Estado inicial
        self.P = 1.0     # Varianza inicial

    def update(self, p_t):
        # Predicción
        s_pred = self.s_hat
        P_pred = self.P + self.sigma_w2

        # Ganancia
        K_t = P_pred / (P_pred + self.sigma_v2)

        # Actualización
        self.s_hat = s_pred + K_t * (p_t - s_pred)
        self.P = (1 - K_t) * P_pred

        return self.s_hat

    def smooth_sequence(self, probs):
        smoothed = []
        # Reset estado
        self.s_hat = 0.0
        self.P = 1.0
        for p in probs:
            s = self.update(p)
            smoothed.append(s)
        return np.array(smoothed)

In [10]:
from sklearn.metrics import f1_score
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

model = AnomalyDetector(input_dim=input_dim, d_model=32, d_state=8, top_k=5).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

epochs = 5
best_f1 = 0.0

for epoch in range(epochs):
    model.train()
    train_loss = 0
    all_preds, all_labels = [], []

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        logits = model(X_batch)
        # Tomamos la predicción del último paso de la ventana
        loss = criterion(logits[:, -1, :], y_batch)
        loss.backward()

        # Clip de gradiente
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()
        preds = torch.argmax(logits[:, -1, :], dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.cpu().numpy())

    train_f1 = f1_score(all_labels, all_preds, average='macro')

    # Validación
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)
            preds = torch.argmax(logits[:, -1, :], dim=-1).cpu().numpy()
            val_preds.extend(preds)
            val_labels.extend(y_batch.numpy())

    val_f1 = f1_score(val_labels, val_preds, average='macro')
    print(f"Epoch {epoch+1}/{epochs} | Loss: {train_loss/len(train_loader):.4f} | Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), 'checkpoint.pt')
        print("  --> Checkpoint guardado")

print("Entrenamiento completado.")

Usando dispositivo: cuda
Epoch 1/5 | Loss: 0.1037 | Train F1: 0.9618 | Val F1: 0.8685
  --> Checkpoint guardado
Epoch 2/5 | Loss: 0.0505 | Train F1: 0.9845 | Val F1: 0.9085
  --> Checkpoint guardado
Epoch 3/5 | Loss: 0.0332 | Train F1: 0.9902 | Val F1: 0.9259
  --> Checkpoint guardado
Epoch 4/5 | Loss: 0.0271 | Train F1: 0.9917 | Val F1: 0.9326
  --> Checkpoint guardado
Epoch 5/5 | Loss: 0.0207 | Train F1: 0.9939 | Val F1: 0.9198
Entrenamiento completado.


In [12]:
# Demostración de uso externo
loaded_model = AnomalyDetector.load('checkpoint.pt', input_dim=input_dim, d_model=32, d_state=8, top_k=5).to(device)

# Simulamos una ventana nueva
nueva_ventana = X_test_w[0:1] # Shape (1, 10, features)
nueva_ventana_tensor = torch.tensor(nueva_ventana).to(device)

prob_anomaly = loaded_model.predict_proba(nueva_ventana_tensor)
# Extraer la probabilidad del último paso de la ventana, ya que el modelo se entrena sobre este.
print(f"Probabilidad de anomalía para la nueva ventana: {prob_anomaly[:, -1].cpu().item():.4f}")

Probabilidad de anomalía para la nueva ventana: 0.9999


In [14]:
from sklearn.metrics import accuracy_score, recall_score, f1_score, mean_absolute_error, mean_squared_error

# Cargar mejor modelo
model.load_state_dict(torch.load('checkpoint.pt'))
model.eval()

# Obtener probabilidades crudas y etiquetas de test
raw_probs = []
true_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        # model.predict_proba devuelve (batch_size, window_size). Tomamos solo el último paso.
        probs = model.predict_proba(X_batch)
        raw_probs.extend(probs[:, -1].cpu().numpy())
        true_labels.extend(y_batch.numpy())

raw_probs = np.array(raw_probs)
true_labels = np.array(true_labels)

# 1. Baseline (Sin Kalman)
baseline_preds = (raw_probs >= 0.5).astype(int)

# 2. Con Kalman
def evaluate_with_kalman(sigma_w2, sigma_v2, raw_probs, true_labels):
    ks = KalmanSmoother(sigma_w2, sigma_v2)
    smoothed_probs = ks.smooth_sequence(raw_probs)
    smoothed_preds = (smoothed_probs >= 0.5).astype(int)

    # Forzamos que las probs suavizadas estén acotadas para MAE/MSE
    smoothed_probs = np.clip(smoothed_probs, 0, 1)

    acc = accuracy_score(true_labels, smoothed_preds)
    rec = recall_score(true_labels, smoothed_preds, average='macro')
    f1 = f1_score(true_labels, smoothed_preds, average='macro')
    mae = mean_absolute_error(true_labels, smoothed_probs)
    mse = mean_squared_error(true_labels, smoothed_probs)
    return acc, rec, f1, mae, mse

# Configuraciones de Kalman (Sensibilidad)
configs = [
    (1e-5, 1e-1), # Baja varianza de proceso (Suavizado agresivo)
    (1e-3, 1e-2), # Equilibrado
    (1e-1, 1e-3)  # Alta varianza de proceso (Seguimiento rápido)
]

# Calcular métricas Baseline
b_acc = accuracy_score(true_labels, baseline_preds)
b_rec = recall_score(true_labels, baseline_preds, average='macro')
b_f1 = f1_score(true_labels, baseline_preds, average='macro')
b_mae = mean_absolute_error(true_labels, raw_probs)
b_mse = mean_squared_error(true_labels, raw_probs)

print(f"Baseline  | Acc: {b_acc:.4f} | Rec: {b_rec:.4f} | F1: {b_f1:.4f} | MAE: {b_mae:.4f} | MSE: {b_mse:.4f}")

for (sw, sv) in configs:
    acc, rec, f1, mae, mse = evaluate_with_kalman(sw, sv, raw_probs, true_labels)
    print(f"Kalman sw={sw}, sv={sv} | Acc: {acc:.4f} | Rec: {rec:.4f} | F1: {f1:.4f} | MAE: {mae:.4f} | MSE: {mse:.4f}")

Baseline  | Acc: 0.9414 | Rec: 0.9325 | F1: 0.9326 | MAE: 0.0593 | MSE: 0.0548
Kalman sw=1e-05, sv=0.1 | Acc: 0.9385 | Rec: 0.9199 | F1: 0.9279 | MAE: 0.0835 | MSE: 0.0449
Kalman sw=0.001, sv=0.01 | Acc: 0.9413 | Rec: 0.9307 | F1: 0.9323 | MAE: 0.0640 | MSE: 0.0494
Kalman sw=0.1, sv=0.001 | Acc: 0.9414 | Rec: 0.9324 | F1: 0.9326 | MAE: 0.0594 | MSE: 0.0547
